# Deepseek: Reasoning vs CoT
This notebook creates side-by-side visualizations comparing the normalized distribution of components over time for reasoning traces and chain-of-thought (CoT) outputs for Deepseek.

In [ ]:
import re
import pandas as pd
import glob
import ast
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import Counter
import scipy.stats as st

FILTER = "*deepseek*" # filter for deepseek, glm etc

# Set up output directory
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the annotation label pattern and keywords grouped by topic
tag_labels = [
    "<INTER>",
    "<CORR>",
    "<ERR_DESC>",
    "<INST>",
    "<ERR_SIM>",
    "<PLAUS>",
    "<CURATE>",
    "<RECON>"
]

# Extract component names
tag_names = [lbl.strip("<>").lower() for lbl in tag_labels]

# Define color palette
# Okabe & Ito color palette
colors = [
    "#0072B2",  # Blue
    "#009E73",  # Green
    "#E69F00",  # Orange
    "#D55E00",  # Red
    "#F0E442",  # Yellow
    "#CC79A7",  # Purple
    "#56B4E9",  # Cyan / Teal
    "#000000"   # Gray
]
color_map = {tag: colors[i] for i, tag in enumerate(tag_names)}

TAG_DISPLAY_NAMES = {
    "inter": "Task Interpretation",
    "corr": "Correct Answer Ref.",
    "err_desc": "Error Description",
    "inst": "Outcome Instantiation",
    "err_sim": "Error Simulation",
    "plaus": "Plausibility Check",
    "curate": "Final Set Curation",
    "recon": "Reconsideration"
}

# Set up styling

BASE_FONT_SIZE = 13
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times", "STIXGeneral", "TeX Gyre Termes"],
    "font.size": BASE_FONT_SIZE * 1.5,
    "axes.titlesize": BASE_FONT_SIZE * 1.5,
    "axes.labelsize": BASE_FONT_SIZE * 1.5,
    "xtick.labelsize": BASE_FONT_SIZE * 1.5,
    "ytick.labelsize": BASE_FONT_SIZE * 1.5,
    "legend.fontsize": BASE_FONT_SIZE * 1.2,
    "figure.titlesize": BASE_FONT_SIZE * 1.5,
})

In [ ]:
# Load and Parse Annotated Data

def load_and_parse_traces(pattern):
    """
    Load annotated traces from CSV files matching the pattern.
    Returns a dictionary with tag names as keys and lists of normalized positions as values.
    """
    positions_by_tag = {tag: [] for tag in tag_names}
    
    parsed_csv_paths = glob.glob(pattern)
    
    for path in parsed_csv_paths:
        df = pd.read_csv(path)
        for idx, row in df.iterrows():
            reasoning = row.get("trace", "")
            seq_str = row.get("annotation_sequence", "")
            
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            reasoning_len = len(reasoning) if reasoning else 1
            
            for pos, tag in seq:
                if tag in positions_by_tag:
                    norm_pos = pos / reasoning_len
                    positions_by_tag[tag].append(norm_pos)
    
    return positions_by_tag

# Load reasoning traces
print("Loading reasoning traces...")
reasoning_positions = load_and_parse_traces(f"eedi_data/joint_results/annotated/{FILTER}-reasoner_*_annot_parsed.csv")
print(f"Loaded reasoning traces with {sum(len(v) for v in reasoning_positions.values())} total component occurrences")

# Load CoT traces
print("Loading CoT traces...")
cot_positions = load_and_parse_traces(f"eedi_data/joint_results/annotated/{FILTER}-cot-*-chat_*_annot_parsed.csv")
print(f"Loaded CoT traces with {sum(len(v) for v in cot_positions.values())} total component occurrences")

In [ ]:
# ============ CONFIGURATION ============
SHOW_ERROR_BARS = False  # Toggle error bars here
bins=np.linspace(0, 1, 6)
# ======================================

def get_mean_and_ci(scores: np.ndarray, confidence: float = 0.95) -> tuple[float, float]:
    """ Mean and confidence interval using t-distribution """
    mean = np.mean(scores)
    sem = st.sem(scores)
    n = len(scores)
    h = sem * st.t.ppf((1 + confidence) / 2, n - 1)
    return mean, h

# Compute data from raw positions
def compute_line_data(positions_by_tag, bins=np.linspace(0, 1, 6)):
    """
    Compute line graph data from component positions.
    Returns raw bin counts for each component and confidence intervals.
    """
    bin_centers = (bins[:-1] + bins[1:]) / 2
    line_data = {}
    line_ci = {}
    
    for tag in tag_names:
        data = np.array(positions_by_tag.get(tag, []))
        bin_counts, _ = np.histogram(data, bins=bins)
        line_data[tag] = bin_counts
        
        # Compute CI for each bin position
        bin_cis = []
        for bin_idx in range(len(bins) - 1):
            bin_mask = (data >= bins[bin_idx]) & (data < bins[bin_idx + 1])
            bin_positions = data[bin_mask]
            if len(bin_positions) > 1:
                _, ci = get_mean_and_ci(bin_positions)
                bin_cis.append(ci)
            else:
                bin_cis.append(0)
        line_ci[tag] = np.array(bin_cis)
    
    return line_data, line_ci, bin_centers

reasoning_line_data, reasoning_line_ci, bin_centers = compute_line_data(reasoning_positions, bins)
cot_line_data, cot_line_ci, _ = compute_line_data(cot_positions, bins)

# ========== VARIANT 1: Normalize by time-bucket (values sum to 1 at each x) ==========
print("Computing Variant 1: Normalize by time-bucket...")

def normalize_by_timebucket(line_data):
    """Normalize so values sum to 1 at each time-bucket."""
    normalized = {}
    n_bins = len(next(iter(line_data.values())))

    for bin_idx in range(n_bins):
        bin_sum = sum(line_data[tag][bin_idx] for tag in tag_names)
        if bin_sum > 0:
            for tag in tag_names:
                if tag not in normalized:
                    normalized[tag] = np.zeros(n_bins)
                normalized[tag][bin_idx] = line_data[tag][bin_idx] / bin_sum
        else:
            for tag in tag_names:
                if tag not in normalized:
                    normalized[tag] = np.zeros(n_bins)
                normalized[tag][bin_idx] = 0
    
    return normalized

def normalize_ci_by_timebucket(line_data, line_ci):
    """Normalize CI by time-bucket proportionally."""
    normalized_ci = {}
    n_bins = len(next(iter(line_data.values())))
    
    for bin_idx in range(n_bins):
        bin_sum = sum(line_data[tag][bin_idx] for tag in tag_names)
        if bin_sum > 0:
            for tag in tag_names:
                if tag not in normalized_ci:
                    normalized_ci[tag] = np.zeros(n_bins)
                # Propagate CI through normalization
                normalized_ci[tag][bin_idx] = line_ci[tag][bin_idx] / bin_sum if bin_sum > 0 else 0
        else:
            for tag in tag_names:
                if tag not in normalized_ci:
                    normalized_ci[tag] = np.zeros(n_bins)
                normalized_ci[tag][bin_idx] = 0
    
    return normalized_ci

reasoning_variant1 = normalize_by_timebucket(reasoning_line_data)
cot_variant1 = normalize_by_timebucket(cot_line_data)
reasoning_variant1_ci = normalize_ci_by_timebucket(reasoning_line_data, reasoning_line_ci) if SHOW_ERROR_BARS else None
cot_variant1_ci = normalize_ci_by_timebucket(cot_line_data, cot_line_ci) if SHOW_ERROR_BARS else None

# Create figure for Variant 1 with shared y-axis
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

# Plot reasoning traces - Variant 1
for tag in tag_names:
    if SHOW_ERROR_BARS and reasoning_variant1_ci:
        ax2.errorbar(bin_centers, reasoning_variant1[tag], yerr=reasoning_variant1_ci[tag],
                    marker='o', linewidth=2, markersize=8, capsize=4,
                    label=TAG_DISPLAY_NAMES.get(tag, tag), color=color_map[tag])
    else:
        ax2.plot(bin_centers, reasoning_variant1[tag], marker='o', linewidth=2, markersize=8,
                label=TAG_DISPLAY_NAMES.get(tag, tag), color=color_map[tag])

ax2.set_xlabel('Normalized Position In Trace', fontsize=BASE_FONT_SIZE*1.5, fontweight='bold')
ax2.set_title('Reasoning Traces', fontsize=BASE_FONT_SIZE*1.5, fontweight='bold')
ax2.set_xlim(bins[0], bins[-1])
ax2.grid(True, alpha=0.3)
#ax2.legend(loc='upper right', fontsize=BASE_FONT_SIZE*1.0, framealpha=0.95)

# fig1.legend(
#     handles=[plt.Line2D([0], [0], color=color_map[tag], marker='o', lw=2, markersize=8,
#                         label=TAG_DISPLAY_NAMES.get(tag, tag)) for tag in tag_names],
#     loc='upper center',
#     bbox_to_anchor=(0.5, 0.87),  # slightly above the top of the plots
#     ncol=len(tag_names)/2,          # spread across the figure
#     fontsize=BASE_FONT_SIZE*1.2,
#     frameon=True,
#     framealpha=0.95
# )

fig1.legend(
    handles=[plt.Line2D([0], [0], color=color_map[tag], marker='o', lw=2, markersize=8,
                        label=TAG_DISPLAY_NAMES.get(tag, tag)) for tag in tag_names],
    loc='center left',             # attach legend to the left side of the bbox
    bbox_to_anchor=(0.97, 0.55),    # move it just outside the right-hand side of the figure
    ncol=1,                        # stack legend items vertically
    fontsize=BASE_FONT_SIZE*1.2,
    frameon=True,
    framealpha=0.95
)

# Plot CoT traces - Variant 1
for tag in tag_names:
    if SHOW_ERROR_BARS and cot_variant1_ci:
        ax1.errorbar(bin_centers, cot_variant1[tag], yerr=cot_variant1_ci[tag],
                    marker='o', linewidth=2, markersize=8, capsize=4,
                    label=TAG_DISPLAY_NAMES.get(tag, tag), color=color_map[tag])
    else:
        ax1.plot(bin_centers, cot_variant1[tag], marker='o', linewidth=2, markersize=8,
                label=TAG_DISPLAY_NAMES.get(tag, tag), color=color_map[tag])

ax1.set_xlabel('Normalized Position In Trace', fontsize=BASE_FONT_SIZE*1.5, fontweight='bold')
ax1.set_ylabel('Share of Strategies', fontsize=BASE_FONT_SIZE*1.5, fontweight='bold')
ax1.set_title('Chain-of-Thought', fontsize=BASE_FONT_SIZE*1.5, fontweight='bold')
ax1.set_xlim(bins[0], bins[-1])
ax1.grid(True, alpha=0.3)

# Add overall title
# fig1.suptitle('Components Over Time', fontsize=BASE_FONT_SIZE*2, fontweight='bold', y=0.95)

plt.tight_layout(w_pad=1.0)
plt.show()

In [ ]:
# Export Figures

# Save Variant 1 - Normalize by time-bucket
out_pdf_v1 = os.path.join(OUTPUT_DIR, "deepseek_reasoning_vs_cot_components_over_time.pdf")
fig1.savefig(out_pdf_v1, bbox_inches="tight", dpi=300)
print(f"Saved Variant 1 PDF: {out_pdf_v1}")

out_png_v1 = os.path.join(OUTPUT_DIR, "deepseek_reasoning_vs_cot_components_over_time.png")
fig1.savefig(out_png_v1, bbox_inches="tight", dpi=300)
print(f"Saved Variant 1 PNG: {out_png_v1}")

In [ ]:
# Summary Statistics

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

# Helper: compute trace-level stats (avg length, presence counts, counts per trace)
from collections import Counter

def compute_trace_stats(pattern):
    paths = glob.glob(pattern)
    total_traces = 0
    total_len = 0
    presence = {tag: 0 for tag in tag_names}
    counts_per_trace = {tag: [] for tag in tag_names}

    for path in paths:
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            trace = row.get("trace", "")
            seq_str = row.get("annotation_sequence", "")
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            total_traces += 1
            total_len += len(trace) if isinstance(trace, str) else 0

            labels_list = [label for _, label in seq if label in tag_names]
            labels_in_trace = set(labels_list)

            for tag in labels_in_trace:
                presence[tag] += 1

            counts = Counter(labels_list)
            for tag in tag_names:
                counts_per_trace[tag].append(counts.get(tag, 0))

    avg_len = (total_len / total_traces) if total_traces > 0 else 0
    presence_pct = {tag: (presence[tag] / total_traces * 100) if total_traces > 0 else 0 for tag in tag_names}
    return total_traces, avg_len, presence, presence_pct, counts_per_trace


# 2) Compute average lengths
reasoning_pattern = f"eedi_data/joint_results/annotated/{FILTER}-reasoner_*_annot_parsed.csv"
cot_pattern = f"eedi_data/joint_results/annotated/{FILTER}-cot-*-chat_*_annot_parsed.csv"

r_total, r_avg_len, r_presence, r_presence_pct, r_counts_per_trace = compute_trace_stats(reasoning_pattern)
ct_total, ct_avg_len, ct_presence, ct_presence_pct, ct_counts_per_trace = compute_trace_stats(cot_pattern)

print("\nAverage trace lengths:")
print(f"  Reasoning traces (n={r_total}): avg length = {r_avg_len:.1f} chars")
print(f"  CoT traces (n={ct_total}): avg length = {ct_avg_len:.1f} chars")

# 2b) Average occurrences per trace (mean ± 95% CI)
print("\nAverage occurrences per trace (mean ± 95% CI):")
print("  Reasoning traces:")
for tag in tag_names:
    arr = np.array(r_counts_per_trace[tag])
    if len(arr) > 1:
        mean, ci = get_mean_and_ci(arr)
    else:
        mean, ci = (np.mean(arr) if len(arr) > 0 else 0), 0
    print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {mean:.2f} ± {ci:.2f}")
print("  CoT traces:")
for tag in tag_names:
    arr = np.array(ct_counts_per_trace[tag])
    if len(arr) > 1:
        mean, ci = get_mean_and_ci(arr)
    else:
        mean, ci = (np.mean(arr) if len(arr) > 0 else 0), 0
    print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {mean:.2f} ± {ci:.2f}")

# 3) Percentages of traces containing at least one occurrence per label
print("\nPercentage of traces containing at least one occurrence (per label):")
print("  Reasoning traces:")
for tag in tag_names:
    pct = r_presence_pct.get(tag, 0)
    print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {pct:.1f}%")
print("  CoT traces:")
for tag in tag_names:
    pct = ct_presence_pct.get(tag, 0)
    print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {pct:.1f}%")

print("\n" + "="*80)

In [ ]:
import plotly.graph_objects as go
import ast
import pandas as pd
import glob
from collections import Counter, defaultdict
import colorsys

all_tags = [lbl.strip("<>").lower() for lbl in tag_labels]


def plot_sankey_tag_transitions(parsed_csv_paths, tag_labels, nr_sankey_steps=3, min_transition_count=1, merge_consecutive: set[str] = {}, min_outgoing_mass=0.1, max_nr_outgoing_edges=None):
    tag_names = [lbl.strip("<>").lower() for lbl in tag_labels if lbl != "unknown"]
    n_tags = len(tag_names)

    # Use same font as matplotlib (serif, Times)
    FONT_FAMILY = "Times New Roman, Times, STIXGeneral, TeX Gyre Termes, serif"
    BASE_FONT_SIZE = 15
    TITLE_FONT_SIZE = int(BASE_FONT_SIZE * 2)
    LEGEND_FONT_SIZE = int(BASE_FONT_SIZE * 2)

    def to_rgb_tuple(color):
        """Convert a color in various formats to an (r,g,b) tuple of ints 0-255."""
        # handle sequence (tuple/list) of floats (0-1) or ints (0-255)
        if isinstance(color, (list, tuple)):
            if max(color) <= 1.0:
                return tuple(int(c*255) for c in color)
            return tuple(int(c) for c in color)

        color = str(color).strip()
        # hex format
        if color.startswith('#'):
            h = color.lstrip('#')
            return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16))

        # rgb(r,g,b) format
        m = re.match(r'rgb\((\d+),(\d+),(\d+)\)', color)
        if m:
            return (int(m.group(1)), int(m.group(2)), int(m.group(3)))

        # fallback to black
        return (0, 0, 0)

    def dim_color_base(color, factor=0.6):
        r, g, b = to_rgb_tuple(color)
        return f'rgb({int(r*factor)},{int(g*factor)},{int(b*factor)})'

    # Collect subsequences
    all_sequences = []
    for path in parsed_csv_paths:
        df = pd.read_csv(path)
        for seq_str in df["annotation_sequence"].fillna(""):
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            labels = [label for _, label in seq if label in tag_names]

            # merge consecutive labels if specified
            labels = [l1 for l1,l2 in zip(labels[:-1], labels[1:]) if ((l1 != l2) or (l1 not in merge_consecutive))] + [labels[-1]]

            if len(labels) >= nr_sankey_steps:
                for i in range(len(labels) - nr_sankey_steps + 1):
                    all_sequences.append(labels[i:i+nr_sankey_steps])

    transitions = defaultdict(Counter)
    for seq in all_sequences:
        for step in range(nr_sankey_steps-1):
            transitions[(step, seq[step])][seq[step+1]] += 1

    # Nodes
    node_labels = []
    node_indices = {}
    node_colors = []
    idx = 0
    for step in range(nr_sankey_steps):
        for tag in tag_names:
            node_labels.append("")  # Remove labels from nodes
            node_indices[(step, tag)] = idx
            node_colors.append(color_map[tag])
            idx += 1

    # Calculate total outgoing mass per source node to determine transparency threshold
    total_outgoing = defaultdict(int)
    for step in range(nr_sankey_steps-1):
        for from_tag in tag_names:
            for to_tag, count in transitions[(step, from_tag)].items():
                if count >= min_transition_count:
                    total_outgoing[(step, from_tag)] += count

    # Identify top N outgoing edges per source node if max_nr_outgoing_edges is set
    top_edges = set()
    if max_nr_outgoing_edges is not None:
        for step in range(nr_sankey_steps-1):
            for from_tag in tag_names:
                # Get all outgoing edges from this node, sorted by count (descending)
                outgoing = sorted(transitions[(step, from_tag)].items(), key=lambda x: x[1], reverse=True)
                # Keep only top max_nr_outgoing_edges
                for to_tag, count in outgoing[:max_nr_outgoing_edges]:
                    if count >= min_transition_count:
                        top_edges.add((step, from_tag, to_tag))

    sources, targets, values, link_colors = [], [], [], []
    for step in range(nr_sankey_steps-1):
        for from_tag in tag_names:
            total_mass = total_outgoing[(step, from_tag)]
            for to_tag, count in transitions[(step, from_tag)].items():
                if count >= min_transition_count:
                    sources.append(node_indices[(step, from_tag)])
                    targets.append(node_indices[(step+1, to_tag)])
                    values.append(count)
                    
                    # Determine color: transparent if not in top edges or below min_outgoing_mass
                    is_transparent = False
                    
                    # Check if link should be transparent based on min_outgoing_mass
                    if total_mass > 0 and count / total_mass < min_outgoing_mass:
                        is_transparent = True
                    
                    # Check if link should be transparent based on max_nr_outgoing_edges
                    if max_nr_outgoing_edges is not None and (step, from_tag, to_tag) not in top_edges:
                        is_transparent = True
                    
                    if is_transparent:
                        link_colors.append("rgba(0,0,0,0)")  # Fully transparent
                    else:
                        # Dim the color for the link based on the base node color
                        link_colors.append(color_map[from_tag])
                        # link_colors.append(dim_color_base(color_map[from_tag], factor=0.6))

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=5,
            thickness=15,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors
        ))])

    # Add invisible scatter traces to create a legend
    for tag in tag_names:
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode='markers',
            marker=dict(size=10, color=color_map[tag]),
            legendgroup=tag,
            showlegend=False,
            name=TAG_DISPLAY_NAMES.get(tag, tag)
        ))

    fig.update_layout(
        title_text="Chain-of-Thought",
        title_x=0.5,
        title_xanchor="center",
        title_y=0.98,
        margin=dict(
            t=40,   # ↓ reduce this (default is ~100)
            b=60,
            l=40,
            r=40,
        ),
        title_yanchor="top",
        title_font=dict(
            family=FONT_FAMILY,
            size=TITLE_FONT_SIZE,
            color="black",
            weight="bold",   # ← fat title
        ),

        font=dict(family=FONT_FAMILY, size=BASE_FONT_SIZE),
        legend_font=dict(family=FONT_FAMILY, size=LEGEND_FONT_SIZE, color="black"),

        # Axis labels (even if axes are visually hidden)
        xaxis=dict(
            title="Step",
            title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY, color="black", weight="bold"),
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            showline=False,
        ),
        yaxis=dict(
            title="Strategy",
            title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY, color="black", weight="bold"),
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            showline=False,
        ),

        height=450,
        width=500,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.write_image("figures/deepseek_cot_common_subprocesses.pdf", width=570, height=450)
    fig.show()

# --- Usage Example ---
parsed_csv_paths = glob.glob(f"eedi_data/joint_results/annotated/{FILTER}-cot-*_annot_parsed.csv")
nr_sankey_steps = 4
min_outgoing_mass = 0.15
plot_sankey_tag_transitions(parsed_csv_paths, all_tags, nr_sankey_steps=nr_sankey_steps, min_transition_count=1, 
                            min_outgoing_mass = min_outgoing_mass, max_nr_outgoing_edges=1000)

In [ ]:
import plotly.graph_objects as go
import ast
import pandas as pd
import glob
from collections import Counter, defaultdict
import colorsys

all_tags = [lbl.strip("<>").lower() for lbl in tag_labels]


def plot_sankey_tag_transitions(parsed_csv_paths, tag_labels, nr_sankey_steps=3, min_transition_count=1, merge_consecutive: set[str] = {}, min_outgoing_mass=0.1, max_nr_outgoing_edges=None):
    tag_names = [lbl.strip("<>").lower() for lbl in tag_labels if lbl != "unknown"]
    n_tags = len(tag_names)

    # Use same font as matplotlib (serif, Times)
    FONT_FAMILY = "Times New Roman, Times, STIXGeneral, TeX Gyre Termes, serif"
    BASE_FONT_SIZE = 15
    TITLE_FONT_SIZE = int(BASE_FONT_SIZE * 2)
    LEGEND_FONT_SIZE = int(BASE_FONT_SIZE * 2)

    def to_rgb_tuple(color):
        """Convert a color in various formats to an (r,g,b) tuple of ints 0-255."""
        # handle sequence (tuple/list) of floats (0-1) or ints (0-255)
        if isinstance(color, (list, tuple)):
            if max(color) <= 1.0:
                return tuple(int(c*255) for c in color)
            return tuple(int(c) for c in color)

        color = str(color).strip()
        # hex format
        if color.startswith('#'):
            h = color.lstrip('#')
            return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16))

        # rgb(r,g,b) format
        m = re.match(r'rgb\((\d+),(\d+),(\d+)\)', color)
        if m:
            return (int(m.group(1)), int(m.group(2)), int(m.group(3)))

        # fallback to black
        return (0, 0, 0)

    def dim_color_base(color, factor=0.6):
        r, g, b = to_rgb_tuple(color)
        return f'rgb({int(r*factor)},{int(g*factor)},{int(b*factor)})'

    # Collect subsequences
    all_sequences = []
    for path in parsed_csv_paths:
        df = pd.read_csv(path)
        for seq_str in df["annotation_sequence"].fillna(""):
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            labels = [label for _, label in seq if label in tag_names]

            # merge consecutive labels if specified
            labels = [l1 for l1,l2 in zip(labels[:-1], labels[1:]) if ((l1 != l2) or (l1 not in merge_consecutive))] + [labels[-1]]

            if len(labels) >= nr_sankey_steps:
                for i in range(len(labels) - nr_sankey_steps + 1):
                    all_sequences.append(labels[i:i+nr_sankey_steps])

    transitions = defaultdict(Counter)
    for seq in all_sequences:
        for step in range(nr_sankey_steps-1):
            transitions[(step, seq[step])][seq[step+1]] += 1

    # Nodes
    node_labels = []
    node_indices = {}
    node_colors = []
    idx = 0
    for step in range(nr_sankey_steps):
        for tag in tag_names:
            node_labels.append("")  # Remove labels from nodes
            node_indices[(step, tag)] = idx
            node_colors.append(color_map[tag])
            idx += 1

    # Calculate total outgoing mass per source node to determine transparency threshold
    total_outgoing = defaultdict(int)
    for step in range(nr_sankey_steps-1):
        for from_tag in tag_names:
            for to_tag, count in transitions[(step, from_tag)].items():
                if count >= min_transition_count:
                    total_outgoing[(step, from_tag)] += count

    # Identify top N outgoing edges per source node if max_nr_outgoing_edges is set
    top_edges = set()
    if max_nr_outgoing_edges is not None:
        for step in range(nr_sankey_steps-1):
            for from_tag in tag_names:
                # Get all outgoing edges from this node, sorted by count (descending)
                outgoing = sorted(transitions[(step, from_tag)].items(), key=lambda x: x[1], reverse=True)
                # Keep only top max_nr_outgoing_edges
                for to_tag, count in outgoing[:max_nr_outgoing_edges]:
                    if count >= min_transition_count:
                        top_edges.add((step, from_tag, to_tag))

    sources, targets, values, link_colors = [], [], [], []
    for step in range(nr_sankey_steps-1):
        for from_tag in tag_names:
            total_mass = total_outgoing[(step, from_tag)]
            for to_tag, count in transitions[(step, from_tag)].items():
                if count >= min_transition_count:
                    sources.append(node_indices[(step, from_tag)])
                    targets.append(node_indices[(step+1, to_tag)])
                    values.append(count)
                    
                    # Determine color: transparent if not in top edges or below min_outgoing_mass
                    is_transparent = False
                    
                    # Check if link should be transparent based on min_outgoing_mass
                    if total_mass > 0 and count / total_mass < min_outgoing_mass:
                        is_transparent = True
                    
                    # Check if link should be transparent based on max_nr_outgoing_edges
                    if max_nr_outgoing_edges is not None and (step, from_tag, to_tag) not in top_edges:
                        is_transparent = True
                    
                    if is_transparent:
                        link_colors.append("rgba(0,0,0,0)")  # Fully transparent
                    else:
                        # Dim the color for the link based on the base node color
                        link_colors.append(color_map[from_tag])
                        # link_colors.append(dim_color_base(color_map[from_tag], factor=0.6))

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=5,
            thickness=15,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors
        ))])

    # Add invisible scatter traces to create a legend
    for tag in tag_names:
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode='markers',
            marker=dict(size=10, color=color_map[tag]),
            legendgroup=tag,
            showlegend=True,
            name=TAG_DISPLAY_NAMES.get(tag, tag)
        ))

    fig.update_layout(
        title_text="Reasoning Traces",
        title_x=0.33,
        title_xanchor="center",
        title_y=0.98,
        margin=dict(
            t=40,   # ↓ reduce this (default is ~100)
            b=60,
            l=40,
            r=40,
        ),
        title_yanchor="top",
        title_font=dict(
            family=FONT_FAMILY,
            size=TITLE_FONT_SIZE,
            color="black",
            weight="bold",   # ← fat title
        ),

        font=dict(family=FONT_FAMILY, size=BASE_FONT_SIZE),
        legend_font=dict(family=FONT_FAMILY, size=LEGEND_FONT_SIZE, color="black"),
        legend=dict(
            traceorder="normal",  # keep the same order
            itemclick="toggle",   # optional
            itemsizing="constant",  # important: prevents Plotly from expanding spacing
            yanchor="top",
            y=1.0,
            xanchor="right",
            x=1.75,               # move it slightly outside right
            font=dict(family=FONT_FAMILY, size=LEGEND_FONT_SIZE, color="black"),
        ),

        # Axis labels (even if axes are visually hidden)
        xaxis=dict(
            title="Step",
            title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY, color="black", weight="bold"),
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            showline=False,
        ),
        yaxis=dict(
            title="Strategy",
            title_font=dict(size=TITLE_FONT_SIZE, family=FONT_FAMILY, color="black", weight="bold"),
            visible=False,
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            showline=False,
        ),

        height=450,
        width=800,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.write_image("figures/deepseek_reasoning_common_subprocesses.pdf", width=800, height=450)
    fig.show()

# --- Usage Example ---
parsed_csv_paths = glob.glob(f"eedi_data/joint_results/annotated/{FILTER}_annot_parsed.csv")
nr_sankey_steps = 4
min_outgoing_mass = 0.15
plot_sankey_tag_transitions(parsed_csv_paths, all_tags, nr_sankey_steps=nr_sankey_steps, min_transition_count=1, 
                            min_outgoing_mass = min_outgoing_mass, max_nr_outgoing_edges=1000)